In [1]:
import sys
import os

print("Python:", sys.executable)
print("Notebook location:", os.getcwd())

Python: C:\New folder\python.exe
Notebook location: C:\Users\kenis


In [3]:
import sys
import os

# Determine project root dynamically
cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, '..')) if os.path.basename(cwd) == 'notebooks' else cwd
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'RAVDESS')
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed')
REPORTS_PATH = os.path.join(PROJECT_ROOT, 'report')

print('Project Root:', PROJECT_ROOT)
print('Raw Dataset Path:', RAW_DATA_PATH)
print('Raw Dataset Exists:', os.path.exists(RAW_DATA_PATH))


Project Root: c:\Users\kamal\group 2\Group-2-Speech-Recognition-Emotion-Classification-
Raw Dataset Path: data/raw/RAVDESS
Raw Dataset Exists: True


In [5]:
print('Dataset exists:', os.path.exists(RAW_DATA_PATH))
actor_dirs = [d for d in os.listdir(RAW_DATA_PATH) if os.path.isdir(os.path.join(RAW_DATA_PATH, d))]
print('Actor directories found:', len(actor_dirs))


Dataset exists: True


In [7]:
wav_files = []
for root, dirs, files in os.walk(RAW_DATA_PATH):
    for file in files:
        if file.lower().endswith('.wav'):
            wav_files.append(os.path.join(root, file))

print('Total WAV files found:', len(wav_files))


Total WAV files found: 1440


In [9]:
emotion_map = {
    1: "neutral",
    2: "calm",
    3: "happy",
    4: "sad",
    5: "angry",
    6: "fearful",
    7: "disgust",
    8: "surprised"
}

print(emotion_map)

{1: 'neutral', 2: 'calm', 3: 'happy', 4: 'sad', 5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised'}


In [11]:
import pandas as pd
import os

records = []

for filepath in wav_files:

    filename = os.path.basename(filepath)

    # Remove .wav and split filename
    parts = filename.replace(".wav", "").split("-")

    # RAVDESS filenames should have 7 parts
    if len(parts) != 7:
        continue

    try:
        modality = int(parts[0])
        vocal_channel = int(parts[1])
        emotion_code = int(parts[2])
        intensity = int(parts[3])
        statement = int(parts[4])
        repetition = int(parts[5])
        actor = int(parts[6])

        records.append({
            "filename": filename,
            "filepath": filepath,
            "modality": modality,
            "vocal_channel": vocal_channel,
            "emotion_code": emotion_code,
            "emotion": emotion_map.get(emotion_code),
            "intensity": intensity,
            "statement": statement,
            "repetition": repetition,
            "actor": actor
        })

    except ValueError:
        continue

df = pd.DataFrame(records)

print("Total audio files:", len(df))
print("Number of actors:", df["actor"].nunique())
print("Number of emotions:", df["emotion"].nunique())

df.head()

Total audio files: 1440
Number of actors: 24
Number of emotions: 8


,filename,filepath,modality,vocal_channel,emotion_code,emotion,intensity,statement,repetition,actor
0,03-01-01-01-01-01-01.wav,C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\r...,3,1,1,neutral,1,1,1,1
1,03-01-01-01-01-02-01.wav,C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\r...,3,1,1,neutral,1,1,2,1
2,03-01-01-01-02-01-01.wav,C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\r...,3,1,1,neutral,1,2,1,1
3,03-01-01-01-02-02-01.wav,C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\r...,3,1,1,neutral,1,2,2,1
4,03-01-02-01-01-01-01.wav,C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\r...,3,1,2,calm,1,1,1,1


In [13]:
print("========== DATASET QUALITY CHECK ==========")

print("\nTotal files:", len(df))

print("\nEmotion distribution:")
print(df["emotion"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate filenames:")
print(df["filename"].duplicated().sum())

print("\nDuplicate file paths:")
print(df["filepath"].duplicated().sum())

print("\nActor distribution:")
print(df["actor"].value_counts().sort_index())

print("\n============================================")

========== DATASET QUALITY CHECK ==========

Total files: 1440

Emotion distribution:
emotion
calm         192
happy        192
sad          192
angry        192
disgust      192
fearful      192
surprised    192
neutral       96
Name: count, dtype: int64

Missing values:
filename         0
filepath         0
modality         0
vocal_channel    0
emotion_code     0
emotion          0
intensity        0
statement        0
repetition       0
actor            0
dtype: int64

Duplicate filenames:
0

Duplicate file paths:
0

Actor distribution:
actor
1     60
2     60
3     60
4     60
5     60
6     60
7     60
8     60
9     60
10    60
11    60
12    60
13    60
14    60
15    60
16    60
17    60
18    60
19    60
20    60
21    60
22    60
23    60
24    60
Name: count, dtype: int64



In [15]:
import librosa
import soundfile as sf

print("Librosa:", librosa.__version__)
print("SoundFile:", sf.__version__)
print("Audio libraries are working!")

Librosa: 1.0.0
SoundFile: 0.14.0
Audio libraries are working!


In [17]:
results = []
errors = []

for index, row in df.iterrows():

    filepath = row["filepath"]

    try:
        # Load one audio file at a time
        y, sr = librosa.load(filepath, sr=None)

        duration = len(y) / sr

        results.append({
            "filename": row["filename"],
            "sampling_rate": sr,
            "duration_seconds": duration,
            "samples": len(y),
            "status": "OK"
        })

    except Exception as e:

        errors.append({
            "filename": row["filename"],
            "error": str(e),
            "status": "ERROR"
        })

quality_df = pd.DataFrame(results)
error_df = pd.DataFrame(errors)

print("========== AUDIO QUALITY CHECK ==========")

print("Files checked:", len(df))
print("Readable files:", len(quality_df))
print("Unreadable files:", len(error_df))

print("\nSampling rates:")
print(quality_df["sampling_rate"].value_counts())

print("\nDuration statistics:")
print(quality_df["duration_seconds"].describe())

print("==========================================")

C:\New folder\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\New folder\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\New folder\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


========== AUDIO QUALITY CHECK ==========
Files checked: 1440
Readable files: 1440
Unreadable files: 0

Sampling rates:
sampling_rate
16000    1440
Name: count, dtype: int64

Duration statistics:
count    1440.000000
mean        3.700685
std         0.336675
min         2.936313
25%         3.470188
50%         3.670375
75%         3.870563
max         5.271937
Name: duration_seconds, dtype: float64


In [19]:
REPORTS_PATH = os.path.join(PROJECT_PATH, "report")

os.makedirs(REPORTS_PATH, exist_ok=True)

quality_path = os.path.join(
    REPORTS_PATH,
    "audio_quality.csv"
)

error_path = os.path.join(
    REPORTS_PATH,
    "corrupted_files.csv"
)

quality_df.to_csv(quality_path, index=False)
error_df.to_csv(error_path, index=False)

print("Audio quality report saved!")
print("Saved:", quality_path)

print("\nCorrupted files report saved!")
print("Saved:", error_path)

Audio quality report saved!
Saved: C:/Users/kenis/OneDrive/Desktop/RAVDESS\reports\audio_quality.csv

Corrupted files report saved!
Saved: C:/Users/kenis/OneDrive/Desktop/RAVDESS\reports\corrupted_files.csv


In [21]:
print("========== DURATION OUTLIER CHECK ==========")

print("\nFiles shorter than 2 seconds:")
short_files = quality_df[quality_df["duration_seconds"] < 2]
print(short_files)

print("\nFiles longer than 6 seconds:")
long_files = quality_df[quality_df["duration_seconds"] > 6]
print(long_files)

print("\nNumber of unusually short files:", len(short_files))
print("Number of unusually long files:", len(long_files))

print("============================================")

========== DURATION OUTLIER CHECK ==========

Files shorter than 2 seconds:
Empty DataFrame
Columns: [filename, sampling_rate, duration_seconds, samples, status]
Index: []

Files longer than 6 seconds:
Empty DataFrame
Columns: [filename, sampling_rate, duration_seconds, samples, status]
Index: []

Number of unusually short files: 0
Number of unusually long files: 0


In [23]:
PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Processed folder ready!")
print("Location:", PROCESSED_PATH)

Processed folder ready!
Location: C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\processed


In [25]:
from sklearn.model_selection import GroupShuffleSplit

# Use actor as the grouping variable
groups = df["actor"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, df["emotion"], groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("========== TRAIN/TEST SPLIT ==========")
print("Training files:", len(train_df))
print("Testing files:", len(test_df))

print("\nTraining actors:")
print(sorted(train_df["actor"].unique()))

print("\nTesting actors:")
print(sorted(test_df["actor"].unique()))

print("======================================")

========== TRAIN/TEST SPLIT ==========
Training files: 1140
Testing files: 300

Training actors:
[np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(18), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]

Testing actors:
[np.int64(1), np.int64(9), np.int64(12), np.int64(17), np.int64(19)]


In [27]:
train_actors = set(train_df["actor"].unique())
test_actors = set(test_df["actor"].unique())

overlap = train_actors.intersection(test_actors)

print("========== LEAKAGE CHECK ==========")
print("Training actors:", len(train_actors))
print("Testing actors:", len(test_actors))
print("Overlapping actors:", overlap)
print("Leakage detected:", len(overlap) > 0)
print("===================================")

========== LEAKAGE CHECK ==========
Training actors: 19
Testing actors: 5
Overlapping actors: set()
Leakage detected: False


In [29]:
train_path = os.path.join(
    PROCESSED_PATH,
    "train_metadata.csv"
)

test_path = os.path.join(
    PROCESSED_PATH,
    "test_metadata.csv"
)

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("========== SPLIT SAVED ==========")
print("Training metadata:", train_path)
print("Testing metadata:", test_path)
print("Training files:", len(train_df))
print("Testing files:", len(test_df))
print("=================================")

========== SPLIT SAVED ==========
Training metadata: C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\processed\train_metadata.csv
Testing metadata: C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\processed\test_metadata.csv
Training files: 1140
Testing files: 300


In [31]:
import numpy as np

def extract_mfcc(filepath, n_mfcc=40):
    """
    Extract 40 MFCC features from one audio file.
    """
    
    y, sr = librosa.load(filepath, sr=None)

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=n_mfcc
    )

    # Average each MFCC across time
    mfcc_mean = np.mean(mfcc, axis=1)

    return mfcc_mean


# Test on one audio file
test_features = extract_mfcc(df.iloc[0]["filepath"])

print("Number of MFCC features:", len(test_features))
print("Feature shape:", test_features.shape)
print("First 5 features:", test_features[:5])

Number of MFCC features: 40
Feature shape: (40,)
First 5 features: [-6.9370599e+02  5.0268967e+01  3.8944486e-01  1.4521958e+01
  3.1664810e+00]


In [33]:
train_features = []
train_labels = []
train_filenames = []

print("Extracting MFCC features from training data...")

for index, row in train_df.iterrows():

    try:
        features = extract_mfcc(row["filepath"])

        train_features.append(features)
        train_labels.append(row["emotion"])
        train_filenames.append(row["filename"])

    except Exception as e:
        print("Error:", row["filename"], e)

train_features = np.array(train_features)

print("\n========== TRAINING FEATURES ==========")
print("Number of files processed:", len(train_features))
print("Feature matrix shape:", train_features.shape)
print("Number of labels:", len(train_labels))
print("=======================================")

Extracting MFCC features from training data...

========== TRAINING FEATURES ==========
Number of files processed: 1140
Feature matrix shape: (1140, 40)
Number of labels: 1140


In [35]:
test_features = []
test_labels = []
test_filenames = []

print("Extracting MFCC features from testing data...")

for index, row in test_df.iterrows():

    try:
        features = extract_mfcc(row["filepath"])

        test_features.append(features)
        test_labels.append(row["emotion"])
        test_filenames.append(row["filename"])

    except Exception as e:
        print("Error:", row["filename"], e)

test_features = np.array(test_features)

print("\n========== TESTING FEATURES ==========")
print("Number of files processed:", len(test_features))
print("Feature matrix shape:", test_features.shape)
print("Number of labels:", len(test_labels))
print("======================================")

Extracting MFCC features from testing data...

========== TESTING FEATURES ==========
Number of files processed: 300
Feature matrix shape: (300, 40)
Number of labels: 300


In [37]:
# Create column names for the 40 MFCC features
mfcc_columns = [f"MFCC_{i+1}" for i in range(40)]

# Training feature DataFrame
train_features_df = pd.DataFrame(
    train_features,
    columns=mfcc_columns
)

train_features_df["emotion"] = train_labels
train_features_df["filename"] = train_filenames

# Testing feature DataFrame
test_features_df = pd.DataFrame(
    test_features,
    columns=mfcc_columns
)

test_features_df["emotion"] = test_labels
test_features_df["filename"] = test_filenames

# Save CSV files
train_features_path = os.path.join(
    PROCESSED_PATH,
    "train_features.csv"
)

test_features_path = os.path.join(
    PROCESSED_PATH,
    "test_features.csv"
)

train_features_df.to_csv(train_features_path, index=False)
test_features_df.to_csv(test_features_path, index=False)

print("========== FEATURE MATRICES SAVED ==========")
print("Training:", train_features_path)
print("Testing:", test_features_path)

print("\nTraining shape:", train_features_df.shape)
print("Testing shape:", test_features_df.shape)
print("============================================")

========== FEATURE MATRICES SAVED ==========
Training: C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\processed\train_features.csv
Testing: C:/Users/kenis/OneDrive/Desktop/RAVDESS\data\processed\test_features.csv

Training shape: (1140, 42)
Testing shape: (300, 42)


In [39]:
print("========== FEATURE QUALITY CHECK ==========")

print("\nTraining missing values:")
print(train_features_df.isnull().sum().sum())

print("\nTesting missing values:")
print(test_features_df.isnull().sum().sum())

print("\nTraining infinite values:")
print(np.isinf(train_features_df[mfcc_columns].values).sum())

print("\nTesting infinite values:")
print(np.isinf(test_features_df[mfcc_columns].values).sum())

print("\nTraining duplicate rows:")
print(train_features_df.duplicated().sum())

print("\nTesting duplicate rows:")
print(test_features_df.duplicated().sum())

print("\n============================================")

========== FEATURE QUALITY CHECK ==========

Training missing values:
0

Testing missing values:
0

Training infinite values:
0

Testing infinite values:
0

Training duplicate rows:
0

Testing duplicate rows:
0



In [41]:
audit_data = {
    "Check": [
        "Expected audio files",
        "Audio files found",
        "Readable audio files",
        "Unreadable audio files",
        "Sampling rate",
        "Number of actors",
        "Number of emotion classes",
        "Missing metadata values",
        "Duplicate filenames",
        "Duplicate file paths",
        "Training files",
        "Testing files",
        "Actor overlap between train/test",
        "Training missing feature values",
        "Testing missing feature values",
        "Training infinite feature values",
        "Testing infinite feature values",
        "Training duplicate feature rows",
        "Testing duplicate feature rows"
    ],

    "Result": [
        1440,
        len(df),
        len(quality_df),
        len(error_df),
        "16 kHz",
        df["actor"].nunique(),
        df["emotion"].nunique(),
        int(df.isnull().sum().sum()),
        int(df["filename"].duplicated().sum()),
        int(df["filepath"].duplicated().sum()),
        len(train_df),
        len(test_df),
        len(train_actors.intersection(test_actors)),
        int(train_features_df.isnull().sum().sum()),
        int(test_features_df.isnull().sum().sum()),
        int(np.isinf(train_features_df[mfcc_columns].values).sum()),
        int(np.isinf(test_features_df[mfcc_columns].values).sum()),
        int(train_features_df.duplicated().sum()),
        int(test_features_df.duplicated().sum())
    ]
}

audit_df = pd.DataFrame(audit_data)

audit_path = os.path.join(
    REPORTS_PATH,
    "data_quality_audit.csv"
)

audit_df.to_csv(audit_path, index=False)

print("========== DATA QUALITY AUDIT ==========")
print(audit_df.to_string(index=False))
print("\nAudit report saved at:")
print(audit_path)
print("========================================")

========== DATA QUALITY AUDIT ==========
                           Check Result
            Expected audio files   1440
               Audio files found   1440
            Readable audio files   1440
          Unreadable audio files      0
                   Sampling rate 16 kHz
                Number of actors     24
       Number of emotion classes      8
         Missing metadata values      0
             Duplicate filenames      0
            Duplicate file paths      0
                  Training files   1140
                   Testing files    300
Actor overlap between train/test      0
 Training missing feature values      0
  Testing missing feature values      0
Training infinite feature values      0
 Testing infinite feature values      0
 Training duplicate feature rows      0
  Testing duplicate feature rows      0

Audit report saved at:
C:/Users/kenis/OneDrive/Desktop/RAVDESS\reports\data_quality_audit.csv


In [1]:
import sys
import os

# Look for src directory in current workspace, data/RAVDESS/src, or Desktop
for candidate in [
    os.path.abspath(os.path.join(os.getcwd(), 'src')),
    os.path.abspath(os.path.join(PROJECT_PATH, 'src')),
    r'PROJECT_ROOT/src'
]:
    if os.path.exists(candidate) and candidate not in sys.path:
        sys.path.append(candidate)

from data_pipeline import run_pipeline

run_pipeline()



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\New folder\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\New folder\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\New folder\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\New folder\Lib\site-packages\tornado\platform\asyncio.py", line 205,

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\New folder\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\New folder\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\New folder\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\New folder\Lib\site-packages\tornado\platform\asyncio.py", line 205,

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\New folder\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\New folder\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\New folder\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\New folder\Lib\site-packages\tornado\platform\asyncio.py", line 205,

...


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\New folder\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\New folder\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\New folder\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\New folder\Lib\site-packages\tornado\platform\asyncio.py", line 205,

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Starting RAVDESS Data Pipeline...
Audio files found: 1440
Metadata records: 1440


C:\New folder\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\New folder\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\New folder\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


Readable files: 1440
Unreadable files: 0
Pipeline completed successfully!
